# LooGLE Head-to-Head: Entroly vs Agentic Pruning
This notebook runs the official Entroly benchmark against the 2026 SOTA Agentic Pruning baseline.

In [ ]:
!pip install -q datasets openai

In [ ]:
import os, sys, subprocess
if not os.path.exists('/content/entroly'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/juyterman1000/entroly.git', '/content/entroly'], check=True)
sys.path.insert(0, '/content/entroly')
from entroly.universal_compress import universal_compress


In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')


In [ ]:
SUBSET = 'shortdep_qa'
SAMPLES = 30
BUDGET = 1500
ANSWER_MODEL = 'gpt-4o-mini'
METHODS = ['baseline', 'entroly', 'agentic_pruning']


In [ ]:
from datasets import load_dataset
ds = load_dataset('bigai-nlco/LooGLE', SUBSET, split='test')
n_avail = len(ds)
indices = list(range(0, n_avail, max(1, n_avail // SAMPLES)))[:SAMPLES]


In [ ]:
def tokens(text: str) -> int:
    return max(1, len(text) // 4)

def compress_truncate(context: str, budget: int) -> str:
    return context[: budget * 4]

def compress_entroly(context: str, budget: int) -> str:
    cur = tokens(context)
    if cur <= budget:
        return context
    target_ratio = max(0.05, (budget * 0.85) / max(cur, 1))
    out, _, _ = universal_compress(context, target_ratio, 'prose')
    if tokens(out) > budget:
        out = out[: budget * 4]
    return out

def compress_agentic_pruner(context: str, question: str, budget: int) -> str:
    cur = tokens(context)
    if cur <= budget:
        return context
    from openai import OpenAI
    client = OpenAI()
    prompt = f"Extract only the exact facts from the following text that are relevant to this question: {question}\n\n{context}"
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=budget,
            temperature=0.0
        )
        return resp.choices[0].message.content.strip()
    except Exception:
        return context[:budget * 4]


In [ ]:
import re, string, time
from collections import Counter, defaultdict
from openai import OpenAI
client = OpenAI()

def answer(context: str, question: str) -> str:
    prompt = f'Use only the context below. Give the shortest correct answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:'
    resp = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=128, temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

def normalize(s: str) -> str:
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    return ' '.join(s.split())

def f1_score(pred: str, gold: str) -> float:
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g: return float(p == g)
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    return 2 * (n / len(p)) * (n / len(g)) / ((n / len(p)) + (n / len(g)))


In [ ]:
stats = {m: defaultdict(float) for m in METHODS}
rows = []

for i, idx in enumerate(indices):
    ex = ds[int(idx)]
    context = ex.get('context') or ex.get('input') or ''
    question = ex.get('question') or ''
    gold = ex.get('answer') or ''
    if isinstance(gold, list): gold = gold[0] if gold else ''
    if not (context and question and gold): continue

    orig_tok = tokens(context)
    for m in METHODS:
        t0 = time.time()
        try:
            if m == 'baseline': ctx = compress_truncate(context, BUDGET)
            elif m == 'entroly': ctx = compress_entroly(context, BUDGET)
            else: ctx = compress_agentic_pruner(context, question, BUDGET)
            cms = (time.time() - t0) * 1000
            
            pred = answer(ctx, question)
            f1 = f1_score(pred, gold)
            
            stats[m]['f1'] += f1
            stats[m]['em'] += 1 if f1 == 1.0 else 0
            stats[m]['tokens_in'] += tokens(ctx)
            stats[m]['original_tokens'] += orig_tok
            stats[m]['ms'] += cms
            stats[m]['n'] += 1
        except Exception as e:
            stats[m]['errors'] += 1
            print(f"Error on {m}: {e}")


In [ ]:
summary = {}
for m in METHODS:
    s = stats[m]
    n = max(1, int(s['n']))
    avg_in = s['tokens_in'] / n
    avg_orig = s['original_tokens'] / n
    
    if m == 'agentic_pruning':
        api_calls = 2
        cost = ((avg_orig * 0.150) + (avg_in * 0.600) + (avg_in * 0.150)) / 1000
    else:
        api_calls = 1
        cost = (avg_in * 0.150) / 1000
        
    summary[m] = {
        'n': int(s['n']),
        'errors': int(s['errors']),
        'avg_f1': round(s['f1'] / n, 3),
        'avg_tokens': round(avg_in, 0),
        'avg_ms': round(s['ms'] / n, 1),
        'api_calls': api_calls,
        'cost_1k': round(cost, 3)
    }

print('\n  LooGLE Head-to-Head')
print('  ' + '─' * 65)
print(f"  {'method':<16} {'F1':>6} {'Tokens':>8} {'Latency':>10} {'API-Calls':>10} {'Cost/1k':>10}")
print('  ' + '─' * 65)
for m in METHODS:
    d = summary[m]
    print(f"  {m:<16} {d['avg_f1']:>6.3f} {int(d['avg_tokens']):>8} {d['avg_ms']:>8.0f}ms {d['api_calls']:>10} ${d['cost_1k']:.3f}")
print('  ' + '─' * 65)
import json
with open('/content/final_results.json', 'w') as f:
    json.dump(summary, f, indent=2)
